# RF-DETR Location Tag — Full Updated Pipeline
This notebook follows the installed RF-DETR `build_roboflow_from_coco()` behavior verified in the user's environment.

**Required layout**
```text
rfdetr_dataset/
├── train/
│   ├── image.jpg
│   └── _annotations.coco.json
├── valid/
│   ├── image.jpg
│   └── _annotations.coco.json
└── test/
    ├── image.jpg
    └── _annotations.coco.json
```
Images are stored directly inside each split directory because RF-DETR passes `root/train`, `root/valid`, and `root/test` directly to `CocoDetection`.

The pipeline automatically discovers classes from `category_id`, groups multiple annotations belonging to the same Azure image URL, downloads images using an Azure connection string from `.env`, creates consistent COCO category mappings, validates paths, trains RF-DETR, evaluates, predicts, and visualizes results.


In [ ]:
# 1. Imports
import os
import json
import hashlib
import random
import shutil
import inspect
from pathlib import Path
from collections import Counter, defaultdict
from urllib.parse import urlparse, unquote

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from dotenv import load_dotenv

print("Imports loaded.")


In [ ]:
# 2. Configuration
INPUT_JSON_DIR = Path("./coco_files")
DATASET_DIR = Path("./rfdetr_dataset")
OUTPUT_DIR = Path("./rfdetr_output")

# Change this to your actual checkpoint if required.
PRETRAINED_WEIGHTS = Path("/home/jupyter/rf-detr-base-coco.pth")

AZURE_CONNECTION_STRING_ENV = "AZURE_STORAGE_CONNECTION_STRING"

IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"
AREA_FIELD = "area"
BBOX_FORMAT = "xywh"

TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
TEST_RATIO = 0.10
RANDOM_SEED = 42

EPOCHS = 50
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 1
RESOLUTION = 560
DEVICE = "cuda"

DOWNLOAD_WORKERS = 16
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

LEARNING_RATE = None
NUM_WORKERS = None
WARMUP_EPOCHS = None

CONFIDENCE_THRESHOLD = 0.40
MAX_VISUALIZATIONS = 20

assert abs(TRAIN_RATIO + VALID_RATIO + TEST_RATIO - 1.0) < 1e-9
assert BBOX_FORMAT.lower() == "xywh"

for split in ["train", "valid", "test"]:
    (DATASET_DIR / split).mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")


In [ ]:
# 3. Load .env and connect to Azure
load_dotenv()

connection_string = os.getenv(AZURE_CONNECTION_STRING_ENV)
if not connection_string:
    raise EnvironmentError(
        f"{AZURE_CONNECTION_STRING_ENV} was not found. "
        "Add it to your .env file."
    )

from azure.storage.blob import BlobServiceClient

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
print("Azure BlobServiceClient created.")


In [ ]:
# 4. Discover annotation JSON files
json_files = sorted(INPUT_JSON_DIR.glob("*.json"))

if not json_files:
    raise FileNotFoundError(
        f"No annotation JSON files found in {INPUT_JSON_DIR.resolve()}"
    )

for _ in tqdm(json_files, desc="Reading JSON files", unit="file"):
    pass

print(f"Found {len(json_files)} JSON file(s).")
for path in json_files:
    print(" -", path)


In [ ]:
# 5. Load annotation records
all_records = []

for json_path in json_files:
    with open(json_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, list):
        records = payload
    elif isinstance(payload, dict) and isinstance(payload.get("annotations"), list):
        records = payload["annotations"]
    else:
        raise ValueError(
            f"Unsupported annotation JSON structure in {json_path}"
        )

    for record in records:
        record = dict(record)
        record["_source_json"] = str(json_path)
        all_records.append(record)

print(f"Total raw annotation records: {len(all_records):,}")


In [ ]:
# 6. Normalize and validate annotations
def normalize_category(value):
    if isinstance(value, list):
        if len(value) != 1:
            raise ValueError(
                f"One bbox cannot be assigned to multiple classes: {value}"
            )
        value = value[0]

    if value is None:
        raise ValueError("category_id is missing.")

    value = str(value).strip()
    if not value:
        raise ValueError("category_id is empty.")

    return value

normalized_records = []
invalid_records = []

for record in tqdm(all_records, desc="Processing annotations", unit="record"):
    try:
        image_url = record.get(IMAGE_FIELD)
        bbox = record.get(BBOX_FIELD)
        category = normalize_category(record.get(CATEGORY_FIELD))

        if not image_url:
            raise ValueError("image_id is missing.")

        if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
            raise ValueError(f"Invalid bbox: {bbox}")

        bbox = [float(value) for value in bbox]
        x, y, width, height = bbox

        if width <= 0 or height <= 0:
            raise ValueError(f"Invalid bbox dimensions: {bbox}")

        normalized_records.append({
            "image_url": str(image_url),
            "category_name": category,
            "bbox_xywh": bbox,
            "area": width * height,
            "source_json": record["_source_json"],
        })
    except Exception as exc:
        invalid_records.append((record, str(exc)))

print(f"Valid annotations: {len(normalized_records):,}")
print(f"Invalid annotations: {len(invalid_records):,}")

if invalid_records:
    print("First invalid annotation:", invalid_records[0][1])


In [ ]:
# 7. Automatically discover classes
classes = []
seen_classes = set()

for record in normalized_records:
    class_name = record["category_name"]
    if class_name not in seen_classes:
        seen_classes.add(class_name)
        classes.append(class_name)

if not classes:
    raise ValueError("No classes were discovered.")

NUM_CLASSES = len(classes)
category_name_to_id = {
    class_name: index + 1
    for index, class_name in enumerate(classes)
}

print(f"NUM_CLASSES = {NUM_CLASSES}")
for category_id, class_name in enumerate(classes, start=1):
    print(f"{category_id}: {class_name}")


In [ ]:
# 8. Group annotations by image URL
image_records = defaultdict(list)

for record in normalized_records:
    image_records[record["image_url"]].append(record)

image_urls = list(image_records.keys())

print(f"Unique images: {len(image_urls):,}")
print(f"Total annotations: {len(normalized_records):,}")


In [ ]:
# 9. Class statistics
class_counts = Counter(
    record["category_name"]
    for record in normalized_records
)

stats_df = pd.DataFrame([
    {
        "category_id": category_name_to_id[class_name],
        "category_name": class_name,
        "annotations": class_counts[class_name],
    }
    for class_name in classes
])

display(stats_df)


In [ ]:
# 10. Azure Blob URL helpers
def parse_blob_url(blob_url):
    parsed = urlparse(blob_url)

    if not parsed.scheme or not parsed.netloc:
        raise ValueError(f"Invalid Azure Blob URL: {blob_url}")

    path_parts = [
        unquote(part)
        for part in parsed.path.lstrip("/").split("/")
        if part
    ]

    if len(path_parts) < 2:
        raise ValueError(f"Could not parse container/blob from: {blob_url}")

    container_name = path_parts[0]
    blob_name = "/".join(path_parts[1:])

    return container_name, blob_name


def make_local_filename(blob_url):
    parsed = urlparse(blob_url)
    original_name = Path(unquote(parsed.path)).name
    extension = Path(original_name).suffix.lower()

    if extension not in IMAGE_EXTENSIONS:
        extension = ".jpg"

    # Full URL is hashed so SAS URLs with different query strings remain unique.
    digest = hashlib.md5(blob_url.encode("utf-8")).hexdigest()

    return f"{digest}{extension}"


In [ ]:
# 11. Download one image
def download_image(blob_url, destination):
    container_name, blob_name = parse_blob_url(blob_url)

    container_client = blob_service_client.get_container_client(container_name)
    blob_client = container_client.get_blob_client(blob_name)

    with open(destination, "wb") as file_handle:
        stream = blob_client.download_blob()
        stream.readinto(file_handle)

    return destination


In [ ]:
# 12. Download all unique images
download_cache = DATASET_DIR / "_downloaded"
download_cache.mkdir(parents=True, exist_ok=True)

downloaded = {}
download_errors = []

for image_url in tqdm(
    image_urls,
    desc="Downloading images",
    unit="image",
):
    filename = make_local_filename(image_url)
    destination = download_cache / filename

    try:
        if not destination.exists() or destination.stat().st_size == 0:
            download_image(image_url, destination)

        with Image.open(destination) as image:
            width, height = image.size

        downloaded[image_url] = {
            "filename": filename,
            "source_path": destination,
            "width": width,
            "height": height,
        }
    except Exception as exc:
        download_errors.append((image_url, str(exc)))

print(f"Downloaded/validated: {len(downloaded):,}")
print(f"Download errors: {len(download_errors):,}")

if download_errors:
    print("First download error:")
    print(download_errors[0][1])


In [ ]:
# 13. Keep only images that downloaded successfully
usable_urls = [
    image_url
    for image_url in image_urls
    if image_url in downloaded
]

if not usable_urls:
    raise RuntimeError("No images were successfully downloaded.")

image_records = {
    image_url: image_records[image_url]
    for image_url in usable_urls
}

print(f"Usable images: {len(image_records):,}")


In [ ]:
# 14. Deterministic image-level split
random.seed(RANDOM_SEED)

split_urls = list(image_records.keys())
random.shuffle(split_urls)

total_images = len(split_urls)

train_end = int(total_images * TRAIN_RATIO)
valid_end = train_end + int(total_images * VALID_RATIO)

train_urls = split_urls[:train_end]
valid_urls = split_urls[train_end:valid_end]
test_urls = split_urls[valid_end:]

print(f"train: {len(train_urls):,}")
print(f"valid: {len(valid_urls):,}")
print(f"test : {len(test_urls):,}")

if len(train_urls) == 0:
    raise RuntimeError("Training split is empty.")


In [ ]:
# 15. Clean generated split directories
for split in ["train", "valid", "test"]:
    split_dir = DATASET_DIR / split

    for item in split_dir.iterdir():
        if item.is_file():
            item.unlink()
        elif item.is_dir():
            shutil.rmtree(item)

print("Generated split directories cleaned.")


In [ ]:
# 16. Copy images directly into train/valid/test
local_image_lookup = {}

for split, urls in {
    "train": train_urls,
    "valid": valid_urls,
    "test": test_urls,
}.items():
    split_dir = DATASET_DIR / split

    for image_url in tqdm(
        urls,
        desc=f"Copying {split} images",
        unit="image",
    ):
        source_path = downloaded[image_url]["source_path"]
        filename = downloaded[image_url]["filename"]
        destination = split_dir / filename

        shutil.copy2(source_path, destination)
        local_image_lookup[image_url] = filename

print("Images copied directly into RF-DETR split directories.")


In [ ]:
# 17. COCO helper functions
def clip_xywh_bbox(bbox, image_width, image_height):
    x, y, width, height = bbox

    x1 = max(0.0, min(float(image_width), x))
    y1 = max(0.0, min(float(image_height), y))
    x2 = max(0.0, min(float(image_width), x + width))
    y2 = max(0.0, min(float(image_height), y + height))

    return [
        x1,
        y1,
        x2 - x1,
        y2 - y1,
    ]


def build_coco_for_split(split, urls):
    split_dir = DATASET_DIR / split

    coco = {
        "info": {
            "description": "RF-DETR custom location tag dataset",
            "version": "1.0",
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": [
            {
                "id": category_name_to_id[class_name],
                "name": class_name,
                "supercategory": "object",
            }
            for class_name in classes
        ],
    }

    next_image_id = 1
    next_annotation_id = 1

    for image_url in urls:
        image_info = downloaded[image_url]

        filename = image_info["filename"]
        image_width = image_info["width"]
        image_height = image_info["height"]

        coco["images"].append({
            "id": next_image_id,
            "file_name": filename,
            "width": image_width,
            "height": image_height,
        })

        for record in image_records[image_url]:
            bbox = clip_xywh_bbox(
                record["bbox_xywh"],
                image_width,
                image_height,
            )

            if bbox[2] <= 0 or bbox[3] <= 0:
                continue

            category_id = category_name_to_id[
                record["category_name"]
            ]

            coco["annotations"].append({
                "id": next_annotation_id,
                "image_id": next_image_id,
                "category_id": category_id,
                "bbox": [round(float(value), 4) for value in bbox],
                "area": round(float(bbox[2] * bbox[3]), 4),
                "iscrowd": 0,
            })

            next_annotation_id += 1

        next_image_id += 1

    annotation_path = split_dir / "_annotations.coco.json"

    with open(annotation_path, "w", encoding="utf-8") as file_handle:
        json.dump(coco, file_handle, indent=2)

    return coco, annotation_path


In [ ]:
# 18. Generate train COCO annotation
train_coco, train_annotation_path = build_coco_for_split(
    "train",
    train_urls,
)

print(train_annotation_path)
print("Images:", len(train_coco["images"]))
print("Annotations:", len(train_coco["annotations"]))


In [ ]:
# 19. Generate valid COCO annotation
valid_coco, valid_annotation_path = build_coco_for_split(
    "valid",
    valid_urls,
)

print(valid_annotation_path)
print("Images:", len(valid_coco["images"]))
print("Annotations:", len(valid_coco["annotations"]))


In [ ]:
# 20. Generate test COCO annotation
test_coco, test_annotation_path = build_coco_for_split(
    "test",
    test_urls,
)

print(test_annotation_path)
print("Images:", len(test_coco["images"]))
print("Annotations:", len(test_coco["annotations"]))


In [ ]:
# 21. Save classes.json
classes_path = DATASET_DIR / "classes.json"

with open(classes_path, "w", encoding="utf-8") as file_handle:
    json.dump(
        {
            "classes": classes,
            "num_classes": NUM_CLASSES,
            "category_name_to_id": category_name_to_id,
        },
        file_handle,
        indent=2,
    )

print(classes_path)


In [ ]:
# 22. Validate exact RF-DETR directory expectations
required_files = [
    DATASET_DIR / "train" / "_annotations.coco.json",
    DATASET_DIR / "valid" / "_annotations.coco.json",
    DATASET_DIR / "test" / "_annotations.coco.json",
]

for required_file in required_files:
    if not required_file.exists():
        raise FileNotFoundError(required_file)

for split in ["train", "valid", "test"]:
    split_dir = DATASET_DIR / split
    annotation_path = split_dir / "_annotations.coco.json"

    with open(annotation_path, "r", encoding="utf-8") as file_handle:
        coco = json.load(file_handle)

    for image in coco["images"]:
        expected_image = split_dir / image["file_name"]

        if not expected_image.exists():
            raise FileNotFoundError(
                f"COCO file_name does not resolve from RF-DETR img_folder: "
                f"{expected_image}"
            )

    print(
        f"{split}: "
        f"{len(coco['images']):,} images, "
        f"{len(coco['annotations']):,} annotations"
    )

print("RF-DETR COCO path validation PASSED.")


In [ ]:
# 23. Validate category mappings
def category_signature(coco):
    return [
        (category["id"], category["name"])
        for category in sorted(
            coco["categories"],
            key=lambda item: item["id"],
        )
    ]

expected_signature = category_signature(train_coco)

if category_signature(valid_coco) != expected_signature:
    raise ValueError("Valid category mapping differs from train.")

if category_signature(test_coco) != expected_signature:
    raise ValueError("Test category mapping differs from train.")

print("Category mapping is identical across train/valid/test.")
print(expected_signature)


In [ ]:
# 24. Dataset statistics
summary_rows = []

for split, coco in [
    ("train", train_coco),
    ("valid", valid_coco),
    ("test", test_coco),
]:
    annotation_counts = Counter(
        annotation["category_id"]
        for annotation in coco["annotations"]
    )

    for class_name in classes:
        category_id = category_name_to_id[class_name]

        summary_rows.append({
            "split": split,
            "category_id": category_id,
            "category_name": class_name,
            "annotations": annotation_counts.get(category_id, 0),
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


In [ ]:
# 25. Inspect installed RF-DETR API
import rfdetr
from rfdetr import RFDETRBase

print("RF-DETR module:", rfdetr.__file__)
print("RFDETRBase:", RFDETRBase)
print("Train signature:", inspect.signature(RFDETRBase.train))

try:
    print("Constructor:", inspect.signature(RFDETRBase))
except Exception as exc:
    print("Constructor signature unavailable:", exc)


In [ ]:
# 26. Verify pretrained weights
if not PRETRAINED_WEIGHTS.exists():
    raise FileNotFoundError(
        f"Pretrained checkpoint not found: {PRETRAINED_WEIGHTS}"
    )

print("Pretrained checkpoint:", PRETRAINED_WEIGHTS)
print(
    "Checkpoint size:",
    round(PRETRAINED_WEIGHTS.stat().st_size / (1024 * 1024), 2),
    "MB",
)


In [ ]:
# 27. Create RF-DETR model with discovered class count
model_kwargs = {
    "num_classes": NUM_CLASSES,
    "pretrain_weights": str(PRETRAINED_WEIGHTS),
    "resolution": RESOLUTION,
    "device": DEVICE,
}

print("Model configuration:")
for key, value in model_kwargs.items():
    print(f"  {key}: {value}")

model = RFDETRBase(**model_kwargs)

print("RF-DETR model initialized.")
print("Model class names:", getattr(model, "class_names", None))


In [ ]:
# 28. Verify model configuration
try:
    model_config = model.get_model_config()
    print(model_config)
except Exception as exc:
    print("Could not print model configuration:", exc)

print("Dataset NUM_CLASSES:", NUM_CLASSES)
print("Dataset classes:", classes)


In [ ]:
# 29. Inspect exact training configuration
from rfdetr.config import get_train_config

print("get_train_config signature:")
print(inspect.signature(get_train_config))

print("\nRFDETRBase.train source:")
print(inspect.getsource(RFDETRBase.train))


In [ ]:
# 30. Build training arguments
train_kwargs = {
    "dataset_dir": str(DATASET_DIR),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "output_dir": str(OUTPUT_DIR),
}

if LEARNING_RATE is not None:
    train_kwargs["lr"] = LEARNING_RATE

if NUM_WORKERS is not None:
    train_kwargs["num_workers"] = NUM_WORKERS

if WARMUP_EPOCHS is not None:
    train_kwargs["warmup_epochs"] = WARMUP_EPOCHS

print("Training arguments:")
for key, value in train_kwargs.items():
    print(f"  {key}: {value}")


In [ ]:
# 31. Validate training configuration before starting
try:
    train_config = get_train_config(**train_kwargs)
    print("Training configuration accepted.")
    print(train_config)
except TypeError:
    print("The installed RF-DETR version rejected a training argument.")
    raise


In [ ]:
# 32. Train
print("Starting RF-DETR training...")
print("Dataset:", DATASET_DIR.resolve())
print("Classes:", classes)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRAD_ACCUM_STEPS)

# RF-DETR/PyTorch Lightning owns its training progress output.
# No artificial tqdm wrapper is used here.
model.train(**train_kwargs)

print("Training completed.")


In [ ]:
# 33. Locate checkpoints
checkpoint_candidates = []

for pattern in ["*.pth", "*.pt", "*.ckpt"]:
    checkpoint_candidates.extend(OUTPUT_DIR.rglob(pattern))

checkpoint_candidates = sorted(
    set(checkpoint_candidates),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

print(f"Found {len(checkpoint_candidates)} checkpoint candidate(s).")

for checkpoint in tqdm(
    checkpoint_candidates,
    desc="Checking checkpoints",
    unit="file",
):
    print(checkpoint)


In [ ]:
# 34. Select trained checkpoint
BEST_CHECKPOINT = None

preferred_names = [
    "checkpoint_best_total.pth",
    "checkpoint_best_ema.pth",
    "checkpoint.pth",
    "last.ckpt",
]

for preferred_name in preferred_names:
    matches = list(OUTPUT_DIR.rglob(preferred_name))

    if matches:
        BEST_CHECKPOINT = max(
            matches,
            key=lambda path: path.stat().st_mtime,
        )
        break

if BEST_CHECKPOINT is None and checkpoint_candidates:
    BEST_CHECKPOINT = checkpoint_candidates[0]

print("Selected checkpoint:", BEST_CHECKPOINT)


In [ ]:
# 35. Reload trained .pth checkpoint
trained_model = None

if BEST_CHECKPOINT is not None and BEST_CHECKPOINT.suffix.lower() == ".pth":
    print("Loading:", BEST_CHECKPOINT)

    try:
        trained_model = RFDETRBase.from_checkpoint(
            str(BEST_CHECKPOINT),
            device=DEVICE,
        )
    except TypeError:
        trained_model = RFDETRBase.from_checkpoint(
            str(BEST_CHECKPOINT)
        )

    print("Trained checkpoint loaded.")
else:
    print("No .pth checkpoint selected.")


In [ ]:
# 36. Inspect evaluation API
if trained_model is not None:
    print("Evaluate signature:")
    print(inspect.signature(trained_model.evaluate))

    print("\nEvaluate source:")
    try:
        print(inspect.getsource(trained_model.evaluate))
    except Exception as exc:
        print("Source unavailable:", exc)


In [ ]:
# 37. Evaluate validation set
if trained_model is None:
    print("No trained model available.")
else:
    try:
        valid_metrics = trained_model.evaluate(
            dataset_dir=str(DATASET_DIR),
            split="valid",
        )
        print(valid_metrics)
    except Exception as exc:
        print("Validation evaluation call failed:")
        print(exc)


In [ ]:
# 38. Evaluate test set
if trained_model is None:
    print("No trained model available.")
else:
    try:
        test_metrics = trained_model.evaluate(
            dataset_dir=str(DATASET_DIR),
            split="test",
        )
        print(test_metrics)
    except Exception as exc:
        print("Test evaluation call failed:")
        print(exc)


In [ ]:
# 39. Inspect prediction API
if trained_model is not None:
    print("predict:", getattr(trained_model, "predict", None))

    if hasattr(trained_model, "predict"):
        try:
            print(inspect.signature(trained_model.predict))
        except Exception as exc:
            print("Prediction signature unavailable:", exc)


In [ ]:
# 40. Run predictions on test images
prediction_results = []

if trained_model is not None:
    test_images = []

    for extension in IMAGE_EXTENSIONS:
        test_images.extend(
            (DATASET_DIR / "test").glob(f"*{extension}")
        )

    test_images = sorted(set(test_images))

    print(f"Test images: {len(test_images):,}")

    for image_path in tqdm(
        test_images,
        desc="Running test predictions",
        unit="image",
    ):
        try:
            result = trained_model.predict(
                str(image_path),
                threshold=CONFIDENCE_THRESHOLD,
            )

            prediction_results.append({
                "image_path": str(image_path),
                "result": result,
            })
        except Exception as exc:
            prediction_results.append({
                "image_path": str(image_path),
                "error": str(exc),
            })

    print(f"Prediction records: {len(prediction_results):,}")
else:
    print("Prediction skipped because trained_model is unavailable.")


In [ ]:
# 41. Inspect first prediction results
for item in prediction_results[:5]:
    print("\nIMAGE:", item["image_path"])

    if "error" in item:
        print("ERROR:", item["error"])
    else:
        print(item["result"])


In [ ]:
# 42. Visualize predictions
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def get_prediction_arrays(result):
    boxes = getattr(result, "xyxy", None)
    confidences = getattr(result, "confidence", None)
    class_ids = getattr(result, "class_id", None)

    if boxes is None:
        boxes = getattr(result, "boxes", None)

    if boxes is None:
        return (
            np.empty((0, 4)),
            np.empty((0,)),
            np.empty((0,), dtype=int),
        )

    boxes = np.asarray(boxes)

    if confidences is None:
        confidences = np.ones(len(boxes))
    else:
        confidences = np.asarray(confidences)

    if class_ids is None:
        class_ids = np.zeros(len(boxes), dtype=int)
    else:
        class_ids = np.asarray(class_ids)

    return boxes, confidences, class_ids


visualized = 0

for item in prediction_results:
    if visualized >= MAX_VISUALIZATIONS:
        break

    if "error" in item:
        continue

    image_path = Path(item["image_path"])

    try:
        image = Image.open(image_path).convert("RGB")
        boxes, confidences, class_ids = get_prediction_arrays(
            item["result"]
        )

        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(image)

        for box, confidence, class_id in zip(
            boxes,
            confidences,
            class_ids,
        ):
            x1, y1, x2, y2 = [float(value) for value in box]

            rectangle = patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                fill=False,
                linewidth=2,
            )
            ax.add_patch(rectangle)

            class_index = int(class_id)

            if 0 <= class_index < len(classes):
                class_name = classes[class_index]
            elif 1 <= class_index <= len(classes):
                class_name = classes[class_index - 1]
            else:
                class_name = str(class_index)

            ax.text(
                x1,
                max(0, y1 - 5),
                f"{class_name} {float(confidence):.2f}",
                fontsize=10,
                bbox={"facecolor": "white", "alpha": 0.7},
            )

        ax.set_title(image_path.name)
        ax.axis("off")
        plt.show()

        visualized += 1

    except Exception as exc:
        print(f"Visualization error for {image_path}: {exc}")


In [ ]:
# 43. Save run metadata
metadata = {
    "classes": classes,
    "num_classes": NUM_CLASSES,
    "category_name_to_id": category_name_to_id,
    "bbox_format": BBOX_FORMAT,
    "train_ratio": TRAIN_RATIO,
    "valid_ratio": VALID_RATIO,
    "test_ratio": TEST_RATIO,
    "random_seed": RANDOM_SEED,
    "resolution": RESOLUTION,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "pretrained_weights": str(PRETRAINED_WEIGHTS),
    "dataset_dir": str(DATASET_DIR),
    "output_dir": str(OUTPUT_DIR),
}

metadata_path = OUTPUT_DIR / "run_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as file_handle:
    json.dump(metadata, file_handle, indent=2)

print("Saved:", metadata_path)


In [ ]:
# 44. Final dataset tree
def print_tree(path, prefix=""):
    path = Path(path)
    entries = sorted(
        path.iterdir(),
        key=lambda item: (item.is_file(), item.name.lower()),
    )

    for index, entry in enumerate(entries):
        connector = "└── " if index == len(entries) - 1 else "├── "
        print(prefix + connector + entry.name)

        if entry.is_dir():
            next_prefix = (
                prefix + "    "
                if index == len(entries) - 1
                else prefix + "│   "
            )
            print_tree(entry, next_prefix)

print_tree(DATASET_DIR)


In [ ]:
# 45. Final validation
print("========== FINAL VALIDATION ==========")
print("Classes:", classes)
print("NUM_CLASSES:", NUM_CLASSES)

for split in ["train", "valid", "test"]:
    annotation_path = DATASET_DIR / split / "_annotations.coco.json"

    with open(annotation_path, "r", encoding="utf-8") as file_handle:
        coco = json.load(file_handle)

    print(
        f"{split}: "
        f"{len(coco['images']):,} images | "
        f"{len(coco['annotations']):,} annotations"
    )

print("Dataset root:", DATASET_DIR.resolve())
print("Output root:", OUTPUT_DIR.resolve())
print("Pretrained:", PRETRAINED_WEIGHTS)
print("======================================")


In [ ]:
# 46. Optional cleanup
DELETE_DOWNLOAD_CACHE = False

if DELETE_DOWNLOAD_CACHE and download_cache.exists():
    shutil.rmtree(download_cache)
    print("Temporary Azure download cache deleted.")
else:
    print("Temporary Azure download cache retained.")


In [ ]:
# 47. Environment information
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

try:
    print("RF-DETR:", rfdetr.__version__)
except Exception:
    print("RF-DETR version attribute unavailable.")


In [ ]:
# 48. Complete
print("RF-DETR Location Tag pipeline completed.")
print("Dataset:", DATASET_DIR.resolve())
print("Training output:", OUTPUT_DIR.resolve())
